# Transit Explorer — North York Town Centre
## Business Intelligence Analysis

**Author:** Tim Nie  
**Date:** June 2026  
**Context:** Portfolio project for Transit App — BI Analyst application

---

### Objective

This notebook explores the point-of-interest (POI) landscape around three TTC subway stations in North York Town Centre:

| Station | Line | Character |
|---------|------|-----------|
| **Finch** | Line 1 Yonge-University | Community & Dining Hub |
| **North York Centre** | Line 1 Yonge-University | Arts, Culture & Civic Core |
| **Sheppard-Yonge** | Line 1 & Line 4 Sheppard | Urban Retail & Connectivity Node |

**Business question:** *What makes each station unique from a visitor discovery perspective, and how can a transit app surface better, transit-aware recommendations?*

In [ ]:
import json
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Style
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

COLORS = {
    'finch': '#1a73e8',
    'north_york_centre': '#34a853',
    'sheppard_yonge': '#ea4335',
}

DATA_DIR = Path('../data')

# Load data
stations_raw = json.loads((DATA_DIR / 'stations.json').read_text())['stations']
pois_raw = json.loads((DATA_DIR / 'pois.json').read_text())['pois']

stations = pd.DataFrame(stations_raw)
pois = pd.DataFrame(pois_raw)

print(f'Loaded {len(stations)} stations and {len(pois)} POIs')
pois.head()

## 1. Dataset Overview

In [ ]:
print('=== Dataset Summary ===')
print(f'Total POIs: {len(pois)}')
print(f'Stations covered: {pois["station_id"].nunique()}')
print(f'Categories: {sorted(pois["category"].unique())}')
print(f'Rating range: {pois["rating"].min():.1f} – {pois["rating"].max():.1f}')
print(f'Review count range: {pois["review_count"].min():,} – {pois["review_count"].max():,}')
print(f'Distance range: {pois["distance_m"].min()}m – {pois["distance_m"].max()}m')
print()

pois_per_station = pois.groupby('station_id').size().reset_index(name='poi_count')
pois_per_station['station_name'] = pois_per_station['station_id'].map({
    'finch': 'Finch',
    'north_york_centre': 'North York Centre',
    'sheppard_yonge': 'Sheppard-Yonge'
})
print('POIs per station:')
print(pois_per_station[['station_name', 'poi_count']].to_string(index=False))

## 2. Category Distribution by Station

Understanding what each station *specializes in* helps a transit app decide which recommendation card types to surface first.

In [ ]:
station_cat = pois.groupby(['station_id', 'category']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))

station_cat.plot(
    kind='bar',
    ax=ax,
    width=0.65,
    colormap='Set2',
    edgecolor='white',
    linewidth=0.8,
)

ax.set_title('POI Category Distribution by Station', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('')
ax.set_ylabel('Number of POIs', fontsize=11)
ax.set_xticklabels(['Finch', 'North York Centre', 'Sheppard-Yonge'], rotation=0, fontsize=11)
ax.legend(title='Category', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('../data/fig_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key insight: Finch leads in restaurants (multicultural dining hub).')
print('North York Centre dominates culture/attractions (Meridian Arts Centre, City Hall, Library).')
print('Sheppard-Yonge has the strongest shopping presence (mall connection, Whole Foods, LCBO).')

## 3. Rating Quality vs. Review Volume

High ratings on few reviews can be misleading. This scatter plot reveals which venues have *both* quality and social proof — the most trustworthy recommendations.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

for station_id, group in pois.groupby('station_id'):
    color = COLORS[station_id]
    station_name = stations.loc[stations['id'] == station_id, 'name'].values[0]
    
    scatter = ax.scatter(
        group['review_count'],
        group['rating'],
        c=color,
        s=group['review_count'] / 30 + 30,
        alpha=0.75,
        label=station_name,
        edgecolors='white',
        linewidth=0.8,
    )
    
    # Label top venues
    top = group.nlargest(2, 'review_count')
    for _, row in top.iterrows():
        ax.annotate(
            row['name'][:20],
            (row['review_count'], row['rating']),
            fontsize=8.5, color='#202124',
            xytext=(6, 2), textcoords='offset points',
        )

# Quadrant: top-right is the "gold zone"
ax.axhline(y=4.2, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax.axvline(x=800, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax.text(850, 4.65, 'Gold Zone\n(high rating + high volume)', fontsize=9, color='gray', alpha=0.7)

ax.set_xlabel('Number of Google Reviews', fontsize=11)
ax.set_ylabel('Average Rating', fontsize=11)
ax.set_title('Rating Quality vs. Review Volume by Station', fontsize=15, fontweight='bold', pad=15)
ax.set_ylim(3.5, 5.0)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/fig_rating_volume.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Walking Distance Distribution

Transit users are highly sensitive to walk time. This histogram shows how the POI landscape concentrates around station exits.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

station_order = ['finch', 'north_york_centre', 'sheppard_yonge']
station_names = ['Finch', 'North York Centre', 'Sheppard-Yonge']

for ax, sid, name in zip(axes, station_order, station_names):
    data = pois[pois['station_id'] == sid]['distance_m']
    color = COLORS[sid]
    
    ax.hist(data, bins=8, color=color, edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.0f}m')
    
    ax.set_title(name, fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('Distance from station (m)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Number of POIs', fontsize=10)
fig.suptitle('Walking Distance Distribution by Station', fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('../data/fig_walk_distance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Insight: Sheppard-Yonge has the most POIs within 100m (mall + Whole Foods directly connected).')
print('Finch has POIs spread over a wider radius, reflecting the larger bus terminal catchment area.')

## 5. Recommendation Score Analysis

The recommendation score weights rating, review volume, and proximity. Here we validate that the scoring surface is sensible and identify which venues the algorithm favours most.

In [ ]:
import sys
sys.path.insert(0, '..')
from app.recommender import score_poi

pois['score'] = pois.apply(
    lambda r: score_poi(r['rating'], r['review_count'], r['distance_m'], r['category']),
    axis=1
)

print('=== Top 10 Recommendations Across All Stations ===')
top10 = pois.nlargest(10, 'score')[['name', 'station_id', 'category', 'rating', 'review_count', 'distance_m', 'score']]
top10['station'] = top10['station_id'].map({'finch': 'Finch', 'north_york_centre': 'NY Centre', 'sheppard_yonge': 'Sheppard-Y'})
top10 = top10.drop('station_id', axis=1)
print(top10.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

top_all = pois.nlargest(15, 'score')
colors = [COLORS[s] for s in top_all['station_id']]

bars = ax.barh(
    top_all['name'].str[:35],
    top_all['score'],
    color=colors,
    edgecolor='white',
    linewidth=0.5,
    height=0.7,
)

for bar, row in zip(bars, top_all.itertuples()):
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height()/2,
            f'★{row.rating}  ({row.review_count:,})',
            va='center', fontsize=8.5, color='#5f6368')

legend_patches = [
    mpatches.Patch(color=c, label=n)
    for n, c in [('Finch', COLORS['finch']),
                 ('North York Centre', COLORS['north_york_centre']),
                 ('Sheppard-Yonge', COLORS['sheppard_yonge'])]
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10)

ax.set_xlabel('Recommendation Score', fontsize=11)
ax.set_title('Top 15 POIs by Recommendation Score', fontsize=14, fontweight='bold', pad=15)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, top_all['score'].max() * 1.25)

plt.tight_layout()
plt.savefig('../data/fig_top_pois.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Score Sensitivity Analysis

How much does distance penalize a venue? This surface plot shows the score landscape for varying rating and distance.

In [ ]:
ratings = np.linspace(3.5, 5.0, 30)
distances = np.linspace(0, 800, 30)
R, D = np.meshgrid(ratings, distances)

# Fixed review count of 500 for this surface
Z = np.vectorize(lambda r, d: score_poi(r, 500, d, 'restaurant'))(R, D)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(R, D, Z, cmap='Blues', edgecolor='none', alpha=0.85)
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Score')

ax.set_xlabel('Rating', fontsize=10, labelpad=8)
ax.set_ylabel('Distance (m)', fontsize=10, labelpad=8)
ax.set_zlabel('Score', fontsize=10)
ax.set_title('Score Surface: Rating vs Distance\n(review_count=500, category=restaurant)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/fig_score_surface.png', dpi=150, bbox_inches='tight')
plt.show()

print('The score surface shows diminishing returns at the distance boundary (~800m).')
print('Rating has larger absolute impact than distance for high-volume venues.')
print('This is intentional: a 4.7★ restaurant 700m away beats a 3.9★ one at the station exit.')

## 7. Station Persona Profiles

Synthesizing the data into actionable, transit-app-ready personas for each station.

In [ ]:
print('\n' + '='*60)
print('STATION PERSONA PROFILES')
print('='*60)

for sid in station_order:
    s_pois = pois[pois['station_id'] == sid]
    station_info = stations[stations['id'] == sid].iloc[0]
    top_cat = s_pois['category'].value_counts().idxmax()
    avg_rating = s_pois['rating'].mean()
    avg_reviews = s_pois['review_count'].mean()
    median_dist = s_pois['distance_m'].median()
    top_venue = s_pois.nlargest(1, 'score').iloc[0]
    
    print(f"\n🚇 {station_info['name'].upper()} ({station_info['character']})")
    print(f"   POIs: {len(s_pois)} | Avg Rating: {avg_rating:.2f}★ | Avg Reviews: {avg_reviews:.0f}")
    print(f"   Median walk: {median_dist:.0f}m | Dominant category: {top_cat}")
    print(f"   #1 Recommendation: {top_venue['name']} (score {top_venue['score']:.2f})")
    print(f"   Transit tip: {station_info['description'][:100]}...")

## 8. Actionable Insights for Transit App

These findings directly translate to product and data decisions a BI Analyst at Transit App would surface.

In [ ]:
insights = [
    {
        'priority': 'HIGH',
        'finding': 'Station character diverges strongly by category',
        'implication': 'Recommendation cards should be dynamically ordered by station persona. '
                       'Finch → lead with Food; North York Centre → lead with Culture/Events; '
                       'Sheppard-Yonge → lead with Shopping + Quick Service.',
    },
    {
        'priority': 'HIGH',
        'finding': 'Sheppard-Yonge has the highest density of POIs within 100m',
        'implication': 'This station is ideal for testing a "at this exit" recommendation feature '
                       'since most high-scoring venues are directly connected to the concourse.',
    },
    {
        'priority': 'MED',
        'finding': 'H-Mart Finch has 2,104 reviews — 5x the average — yet is not #1 by score',
        'implication': 'Review volume alone is a poor ranking signal. The composite score (rating × '
                       'log(reviews) × proximity) outperforms raw sort by review_count for '
                       'tourist-appropriate curation.',
    },
    {
        'priority': 'MED',
        'finding': 'Culture venues (Meridian Arts Centre, Toronto Public Library) score highly',
        'implication': 'A "hidden gems" tab weighted toward culture + parks would differentiate '
                       'Transit App from Google Maps. These venues are underrepresented in '
                       'typical food-first recommendation feeds.',
    },
    {
        'priority': 'LOW',
        'finding': 'Free venues (parks, library, Mel Lastman Square) cluster around 4.3–4.6★',
        'implication': 'Price-level = 0 is a strong positive signal for tourist satisfaction. '
                       'Adding a "Free to visit" filter would increase engagement for cost-sensitive '
                       'transit users.',
    },
]

for i, ins in enumerate(insights, 1):
    print(f"[{ins['priority']}] Insight {i}: {ins['finding']}")
    print(f"        → {ins['implication']}")
    print()

## 9. Next Steps & Data Limitations

**What this analysis does not yet include:**

- **TTC ridership data** — integrating daily boardings per station would let us weight recommendations by congestion (e.g., avoid slow food spots during AM rush at Sheppard-Yonge)
- **Temporal signals** — hours of operation, peak times from Google Places, and TTC schedule data would enable time-aware recommendations ("open now" filter, "5-minute eat before your train")
- **User intent segmentation** — tourists vs. daily commuters vs. weekend explorers have different needs; a Transit App session's context (trip planning vs. local discovery) should modulate which recommendations surface
- **Real-time events** — Mel Lastman Square hosts concerts, which dramatically changes foot traffic patterns around North York Centre station

**Data quality notes:**
- Dataset is curated (28 POIs), not a complete scrape. A production pipeline would collect 200+ venues per station via the Google Places Nearby Search API with pagination.
- Ratings reflect a snapshot in time. A production BI system would track rating drift over 90-day windows.
- Distance is straight-line (Haversine). Production should use Google Distance Matrix API for walk-time via pedestrian routing.

In [ ]:
print('Analysis complete. All figures saved to ../data/')
print('Run the Flask app for the interactive map: python run.py')